# S&P 500 Constituents

Scrapes the S&P 500 Index Components from [slickcharts.com/sp500](https://www.slickcharts.com/sp500) and saves the result to `sp500_constituents/`.

**Output schema**

| Column | Description |
|--------|-------------|
| `rank` | Index rank by market cap weight |
| `company` | Full company name |
| `symbol` | Ticker symbol |
| `weight_pct` | Index weight (%) |
| `price` | Last price (USD) |
| `change` | Price change (USD) |
| `pct_change` | Price change (%) |
| `date` | Snapshot date (ISO 8601) |

In [1]:
import re
from datetime import date
from io import StringIO
from pathlib import Path

import pandas as pd
import requests

OUTPUT_DIR = Path("sp500_constituents")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
URL = "https://www.slickcharts.com/sp500"

# slickcharts blocks the default Python user-agent; spoof a browser header
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(URL, headers=HEADERS, timeout=30)
response.raise_for_status()
print(f"HTTP {response.status_code} — fetched {len(response.content):,} bytes")

HTTP 200 — fetched 369,095 bytes


In [3]:
# pandas.read_html parses every <table>; the first one is the constituents table.
# Wrap in StringIO to avoid the FutureWarning from pandas.
tables = pd.read_html(StringIO(response.text))
print(f"Found {len(tables)} table(s) on the page")

df = tables[0].copy()
print(f"Shape: {df.shape}")
df.head()

Found 3 table(s) on the page
Shape: (503, 7)


,#,Company,Symbol,Weight,Â Â Â Â Â Â Price,Chg,% Chg
0,1,Nvidia,NVDA,7.11%,Â Â 182.48,5.29,(2.99%)
1,2,Apple Inc.,AAPL,6.23%,Â Â 264.72,0.54,(0.20%)
2,3,Microsoft,MSFT,4.75%,Â Â 398.55,5.81,(1.48%)
3,4,Amazon,AMZN,3.59%,Â Â 208.39,-1.61,(-0.77%)
4,5,Alphabet Inc. (Class A),GOOGL,3.07%,Â Â 306.52,-5.24,(-1.68%)


In [4]:
# ---- normalise column names ----
# slickcharts injects non-breaking spaces into the Price header via CSS padding;
# strip all whitespace and non-ASCII junk before renaming.
df.columns = [
    re.sub(r"[^\x00-\x7F]+", "", c).strip()
    for c in df.columns
]
print("Cleaned columns:", df.columns.tolist())

rename_map = {
    "#": "rank",
    "Company": "company",
    "Symbol": "symbol",
    "Weight": "weight_pct",
    "Price": "price",
    "Chg": "change",
    "% Chg": "pct_change",
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
print("Renamed columns:", df.columns.tolist())

Cleaned columns: ['#', 'Company', 'Symbol', 'Weight', 'Price', 'Chg', '% Chg']
Renamed columns: ['rank', 'company', 'symbol', 'weight_pct', 'price', 'change', 'pct_change']


In [5]:
# ---- coerce numeric columns ----
for col in ["weight_pct", "price", "change", "pct_change"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[%$,()]", "", regex=True)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---- snapshot date ----
df["date"] = date.today().isoformat()

print(df.dtypes)
df.head(5)

rank            int64
company        object
symbol         object
weight_pct    float64
price         float64
change        float64
pct_change    float64
date           object
dtype: object


,rank,company,symbol,weight_pct,price,change,pct_change,date
0,1,Nvidia,NVDA,7.11,NaN,5.29,2.99,2026-03-02
1,2,Apple Inc.,AAPL,6.23,NaN,0.54,0.20,2026-03-02
2,3,Microsoft,MSFT,4.75,NaN,5.81,1.48,2026-03-02
3,4,Amazon,AMZN,3.59,NaN,-1.61,-0.77,2026-03-02
4,5,Alphabet Inc. (Class A),GOOGL,3.07,NaN,-5.24,-1.68,2026-03-02


In [6]:
# ---- save (monthly cadence: YYYYMM) ----
month_str = date.today().strftime("%Y%m")
csv_path = OUTPUT_DIR / f"sp500_constituents_{month_str}.csv"

df.to_csv(csv_path, index=False)
print(f"Saved {len(df)} rows → {csv_path}")

Saved 503 rows → sp500_constituents/sp500_constituents_202603.csv


In [7]:
# ---- sanity checks ----
print(f"Total constituents : {len(df)}")
print(f"Sum of weights     : {df['weight_pct'].sum():.2f}%")
print(f"Date snapshot      : {df['date'].iloc[0]}")

print("\nTop 10 by weight:")
display(
    df[["rank", "company", "symbol", "weight_pct", "price"]]
    .sort_values("weight_pct", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Total constituents : 503
Sum of weights     : 99.93%
Date snapshot      : 2026-03-02

Top 10 by weight:


,rank,company,symbol,weight_pct,price
0,1,Nvidia,NVDA,7.11,NaN
1,2,Apple Inc.,AAPL,6.23,NaN
2,3,Microsoft,MSFT,4.75,NaN
3,4,Amazon,AMZN,3.59,NaN
4,5,Alphabet Inc. (Class A),GOOGL,3.07,NaN
5,6,Alphabet Inc. (Class C),GOOG,2.87,NaN
6,7,Meta Platforms,META,2.65,NaN
7,8,"Tesla, Inc.",TSLA,2.43,NaN
8,9,Broadcom,AVGO,2.42,NaN
9,10,Berkshire Hathaway,BRK.B,1.66,NaN
